In [ ]:
# ── CELL 1: INSTALLATION ──────────────────────────────────
# Estimated time: 2-3 min. Wait for "DONE".

import os, shutil, subprocess, sys

# 1. Install system dependencies
!apt-get install -y zstd curl lsof -q 2>/dev/null

# 2. Install Ollama
OLLAMA_PATHS = [
    shutil.which("ollama"),
    "/usr/local/bin/ollama",
    "/usr/bin/ollama",
    os.path.expanduser("~/.local/bin/ollama"),
    os.path.expanduser("~/ollama"),
]
if not any(p and os.path.exists(p) for p in OLLAMA_PATHS):
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("  ✅ Ollama already installed")

# 3. Install Open WebUI
try:
    import open_webui
    print("  ✅ Open WebUI already installed")
except ImportError:
    !pip install open-webui -q

# 4. Install pyngrok (ngrok Python wrapper)
try:
    import pyngrok
    print("  ✅ pyngrok already installed")
except ImportError:
    !pip install pyngrok -q

# 5. Verify all dependencies
missing = []
if not shutil.which("ollama"):
    ollama_found = any(p and os.path.exists(p) for p in OLLAMA_PATHS)
    if not ollama_found:
        missing.append("ollama")
try:
    import pyngrok
except ImportError:
    missing.append("pyngrok")
try:
    import open_webui
except ImportError:
    missing.append("open-webui")

if missing:
    print(f"❌ FAILED: {', '.join(missing)} not found after installation")
else:
    print("────────────────────────────────")
    print("✅ DONE — All dependencies installed. Proceed to Cell 2.")
    print("────────────────────────────────")

In [ ]:
# ── CELL 2: SETUP & RUN OLLAMA ────────────────────────────
# Change model below if needed:
MODEL = "qwen2.5-coder:7b"   # default (~4.7GB)
# MODEL = "qwen2.5-coder:14b"  # coding (~9.0GB)
# MODEL = "huihui_ai/qwen2.5-coder-abliterate:14b-instruct-q4_k_m"  # uncensored coding (~9.0GB)
# MODEL = "supergoatscriptguy/mythos-sec:24b"  # pentest/CTF uncensored (~14GB)
# MODEL = "dolphin-llama3:8b"  # uncensored general/agentic (~4.7GB)
# MODEL = "deepseek-coder:14b" # coding (~8.5GB)
# MODEL = "deepseek-r1:7b"     # reasoning (~4.7GB)
# MODEL = "gemma4:12b"         # reasoning & coding (~6.6GB)
# MODEL = "llama3.1:8b"        # general chat (~4.9GB)

import subprocess, time, os, shutil, json, urllib.request, sys
from google.colab import drive

# ─── Find ollama binary ───
OLLAMA_CANDIDATES = [
    shutil.which("ollama"),
    "/usr/local/bin/ollama",
    "/usr/bin/ollama",
    os.path.expanduser("~/.local/bin/ollama"),
    os.path.expanduser("~/ollama"),
]
ollama_bin = next((p for p in OLLAMA_CANDIDATES if p and os.path.exists(p)), None)
if not ollama_bin:
    print("⚠️  Ollama binary not found — attempting to install...")
    ret = os.system("curl -fsSL https://ollama.com/install.sh | sh")
    if ret != 0:
        print("❌ Ollama install failed. Run Cell 1 manually first.")
        sys.exit(1)
    ollama_bin = next((p for p in OLLAMA_CANDIDATES if p and os.path.exists(p)), None)
    if not ollama_bin:
        print("❌ Ollama installed but binary not found in expected paths.")
        sys.exit(1)
    print("  ✅ Ollama installed successfully")

# ─── Mount Google Drive (only if not already) ───
print("[1/5] Mounting Google Drive...", flush=True)
if not os.path.exists("/content/drive"):
    drive.mount('/content/drive')
else:
    print("  ✅ Drive already mounted")

# ─── Set up model paths ───
print("[2/5] Setting up model folder...")
DRIVE_PATH = "/content/drive/MyDrive/ollama_models"
LOCAL_PATH = os.path.expanduser("~/.ollama/models")

# Restore model from Drive to local (if exists)
RESTORED_FLAG = os.path.join(LOCAL_PATH, ".restored")
if os.path.exists(DRIVE_PATH) and os.listdir(DRIVE_PATH):
    os.makedirs(LOCAL_PATH, exist_ok=True)
    if not os.path.exists(RESTORED_FLAG):
        print("  📦 Restoring model from Google Drive...")
        for item in os.listdir(DRIVE_PATH):
            src = os.path.join(DRIVE_PATH, item)
            dst = os.path.join(LOCAL_PATH, item)
            if not os.path.exists(dst):
                try:
                    if os.path.isdir(src):
                        shutil.copytree(src, dst, dirs_exist_ok=True)
                    else:
                        shutil.copy2(src, dst)
                except Exception as e:
                    print(f"    ⚠️  Failed to copy {item}: {e}")
        open(RESTORED_FLAG, 'w').close()
        print("  ✅ Model restored from Drive")
    else:
        print("  ✅ Model already restored (skip)")
elif not os.path.exists(LOCAL_PATH):
    os.makedirs(LOCAL_PATH, exist_ok=True)

# Use local path (not Drive directly) — more compatible with Ollama
os.environ["OLLAMA_MODELS"] = LOCAL_PATH

# ─── Kill stale Ollama processes (if any) ───
print("[3/5] Cleaning up stale processes...")
os.system("pkill -f 'ollama serve' 2>/dev/null || true")
time.sleep(1)

# ─── Start Ollama ───
print("[4/5] Starting Ollama & loading model...")

# Bind to 127.0.0.1 — more secure, tunnel still works
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
proc_ollama = subprocess.Popen(
    [ollama_bin, "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Wait for Ollama to be ready
print("  Waiting for Ollama to be ready...", end="", flush=True)
ollama_ready = False
for _ in range(30):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        r.close()
        print(" ✅")
        ollama_ready = True
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(2)
if not ollama_ready:
    print(" ⚠️  Ollama not responding — check logs above")

# Pull model
print(f"  Downloading model {MODEL} (may take several minutes)...")
pull = subprocess.Popen(
    [ollama_bin, "pull", MODEL],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in iter(pull.stdout.readline, ''):
    line = line.rstrip()
    if line:
        print(f"  {line}")
pull.wait()

if pull.returncode != 0:
    print(f"  ⚠️  Pull failed (exit code {pull.returncode})")
else:
    print("  ✅ Model pulled successfully")

# Backup model to Drive
if os.path.exists(LOCAL_PATH):
    print("  💾 Backing up model to Google Drive...")
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for item in os.listdir(LOCAL_PATH):
        if item.startswith('.'):
            continue
        src = os.path.join(LOCAL_PATH, item)
        dst = os.path.join(DRIVE_PATH, item)
        if not os.path.exists(dst):
            try:
                if os.path.isdir(src):
                    shutil.copytree(src, dst, dirs_exist_ok=True)
                else:
                    shutil.copy2(src, dst)
            except Exception as e:
                print(f"    ⚠️  Failed to backup {item}: {e}")
    print("  ✅ Backup to Drive complete")

print("────────────────────────────────")
print(f"✅ MODEL {MODEL} READY — Proceed to Cell 3.")
print("────────────────────────────────")

In [ ]:
# ── CELL 3+4: WEBUI + TUNNEL (WEBUI + OLLAMA API) ─────────
# THIS CELL MUST KEEP RUNNING WHILE USING WEBUI / OPENCODE
import os, subprocess, time, urllib.request, re, signal, shutil, sys

WEBUI_PORT = 8081
OLLAMA_PORT = 11434

# ─── Helper: kill process on a given port ───
def kill_port(port):
    methods = [
        f"fuser -k {port}/tcp 2>/dev/null",
        f"lsof -ti:{port} | xargs kill -9 2>/dev/null",
        f"ss -tlnp | grep ':{port}' | tr -s ' ' | cut -d' ' -f7 | cut -d',' -f2 | xargs kill -9 2>/dev/null"
    ]
    for cmd in methods:
        ret = os.system(cmd)
        if os.WIFEXITED(ret) and os.WEXITSTATUS(ret) == 0:
            return True
    return False

# ────────────────────────────────────────────────────────────
#  1. START OPEN WEBUI
# ────────────────────────────────────────────────────────────
webui_bin = shutil.which("open-webui") or shutil.which("open_webui")
if not webui_bin:
    print("❌ open-webui not found. Run Cell 1 first.", flush=True)
    raise SystemExit(1)

print(f"[1/6] Cleaning up port {WEBUI_PORT}...", flush=True)
kill_port(WEBUI_PORT)
time.sleep(1)

print(f"[2/6] Starting Open WebUI on port {WEBUI_PORT}...", flush=True)
env = {**os.environ, "OLLAMA_BASE_URL": "http://127.0.0.1:11434", "WEBUI_AUTH": "False"}
proc_webui = subprocess.Popen(
    [webui_bin, "serve", "--port", str(WEBUI_PORT)],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

print("[3/6] Waiting for WebUI to be ready (initial load 1-3 min)...", flush=True)
webui_ready = False
for i in range(60):
    if proc_webui.poll() is not None:
        print(f"\n❌ WebUI failed to start (exit code {proc_webui.returncode})", flush=True)
        raise SystemExit(1)
    try:
        r = urllib.request.urlopen(f"http://127.0.0.1:{WEBUI_PORT}", timeout=5)
        r.close()
        print("\n✅ WebUI is running!", flush=True)
        webui_ready = True
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(5)

if not webui_ready:
    print("\n⚠️  WebUI timeout — tunnel will still be attempted.", flush=True)

# ────────────────────────────────────────────────────────────
#  2. TUNNEL WITH NGROK (WebUI + Ollama API via proxy)
# ────────────────────────────────────────────────────────────
print("[4/6] Setting up ngrok tunnel...", flush=True)

from pyngrok import ngrok

# ─── NGROK AUTH TOKEN ────────────────────────────────
# Get your free token at https://dashboard.ngrok.com/get-started/your-authtoken
# Paste it between the quotes below:
NGROK_AUTH_TOKEN = ""

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("  ✅ ngrok authenticated")
else:
    print("  ⚠️  No token set — set NGROK_AUTH_TOKEN above or tunnel may show warning")

print("⏳ Creating ngrok tunnels...", end="", flush=True)
try:
    tunnel_webui = ngrok.connect(WEBUI_PORT, "http")
    tunnel_ollama = ngrok.connect(OLLAMA_PORT, "http")
    webui_url = tunnel_webui.public_url.replace("http://", "https://")
    ollama_url = tunnel_ollama.public_url.replace("http://", "https://")
    print(" ✅")
except Exception as e:
    print(f"\n❌ ngrok failed: {e}")
    print("   Make sure your auth token is correct and internet is connected.")
    sys.exit(1)

print("[5/6] Testing tunnels...")
try:
    r = urllib.request.urlopen(f"{webui_url}", timeout=5)
    r.close()
    print("  ✅ WebUI accessible via ngrok")
except Exception:
    print("  ⚠️  WebUI not yet accessible — may take a moment")
try:
    r = urllib.request.urlopen(f"{ollama_url}/api/tags", timeout=5)
    r.close()
    print("  ✅ Ollama API accessible via ngrok (direct tunnel)")
except Exception:
    print("  ⚠️  Ollama API not yet accessible — may take a moment")

print()
print("═" * 60)
print(" 🌐 OPEN WEBUI:")
print(f" 👉 {webui_url}")
print("═" * 60)
print()
print("═" * 60)
print("🔗 OLLAMA API (for opencode):")
print(f"   {ollama_url}")
print("═" * 60)
print()

print("opencode.json configuration:")
print("═" * 60)
print("Create opencode.json in your project folder:\n")
print("{")
print('  "$schema": "https://opencode.ai/config.json",')
print('  "provider": {')
print('    "Ollama": {')
print('      "npm": "@ai-sdk/openai-compatible",')
print('      "name": "Ollama (Colab)",')
print('      "options": {')
print(f'        "baseURL": "{ollama_url}"')
print('      },')
print('      "models": {')
print(f'        "QwenCoder7B": {{')
print(f'          "name": "{MODEL}"')
print('        }')
print('      }')
print('    }')
print('  }')
print("}")
print()
print("📝 Notes:")
print("   - The ngrok tunnel MUST be running for opencode to connect.")
print("   - If Colab disconnects, re-run Cell 1 → Cell 2 → Cell 3+4.")
print("   - Change model to match the one selected in Cell 2.")
print("   - ⚠️  These URLs are PUBLIC — do not share them!")
print("   - Run /models in opencode to see all available models.")
print("   - Ollama API uses a DIRECT tunnel (port 11434), not via WebUI proxy.")

# ────────────────────────────────────────────────────────────
#  KEEP-ALIVE: MONITOR ALL PROCESSES
# ────────────────────────────────────────────────────────────
print("\n📡 Monitoring WebUI + Ollama + ngrok tunnels (Ctrl+C to stop)...")
try:
    while True:
        time.sleep(30)
        if proc_webui.poll() is not None:
            print("⚠️  WebUI process has stopped! Re-run this cell.")
            break
        if not ngrok.get_ngrok_process().proc.poll() is None:
            print("⚠️  ngrok tunnel disconnected! Re-run this cell.")
            break
        print("✓ All services active —", time.strftime("%H:%M:%S"))
except KeyboardInterrupt:
    print("\nAll processes stopped.")